# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

# !git clone https://github.com/IgnacioOQ/e_network_inequality
!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

/content/e_network_inequality/e_network_inequality


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/'
print("Current Directory:", dumping_path)

Mounted at /content/drive
Current Directory: /content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/


In [ ]:
def generate_parameters_here(_,G,method='randomization'):
    process_seed = int.from_bytes(os.urandom(4), byteorder='little')
    rd.seed(process_seed)
    # Randomly sample parameters for this group
    uncertainty = rd.uniform(.000001, .001)
    n_experiments = rd.randint(1000, 10000)
    # now we pick a random number
    # Capped at 1/3 to prevent "Sample larger than population" errors in equalize
    proportion_edges = rd.rand() * (1/3)
    # Do randomization
    num_edges = G.number_of_edges()
    if method == 'randomization':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = randomize_network(G, n_edges=num_edges_to_randomize)
    if method == 'equalize':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = equalize(G, num_edges_to_randomize)
    if method == 'densify':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='original',keep_density_fixed=False)
    if method == 'densify_fixed':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='uniform',keep_density_fixed=True)
    if method =='cluster':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = cluster_network(G,num_edges_to_add)
    if method =='decluster':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = decluster_network(G,num_edges_to_randomize)

    result = generate_parameters_aggregate(modified_network, uncertainty=uncertainty, n_experiments=n_experiments,
                                           p_rewiring=proportion_edges)
    result['uncertainty'] = uncertainty
    result['n_experiments'] = n_experiments
    result['proportion_edges'] = proportion_edges
    result['parameter_random_seed']= process_seed
    return result

# Study 0: Load Networks

In [ ]:
# ── Load Networks ────────────────────────────────────────────────────────────
from model.vectorized_model import VectorizedModel
import itertools
import pickle
import networkx as nx

with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
    G_pud = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_pud.nodes())}
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)
print(f"PUD network: {G_pud_indexed.number_of_nodes()} nodes, {G_pud_indexed.number_of_edges()} edges")

with open('./networks/citation_data/tobacco_network.pkl', 'rb') as f:
    G_tobacco = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_tobacco.nodes())}
G_tobacco_indexed = nx.relabel_nodes(G_tobacco, mapping)
print(f"Tobacco network: {G_tobacco_indexed.number_of_nodes()} nodes, {G_tobacco_indexed.number_of_edges()} edges")

# Study 1: Convergence Speed Analysis

In [ ]:
# ── Convergence Speed Analysis ───────────────────────────────────────────────
# Ported from convergence_speed_analysis.py  (STOPPING_CONDITION_ANALYSIS.md §4)
# Sweeps tolerance ∈ {1e-3, 1e-4, 1e-5, 1e-6}
# Records steps_taken; produces box plots and stopping-time ratio charts.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CS_N_EXPERIMENTS = 100
CS_UNCERTAINTY   = 0.001
CS_N_RUNS        = 100
CS_MAX_STEPS     = 100_000
CS_DEFAULT_TOL   = 1e-3
CS_TOLERANCES    = [1e-3, 1e-4, 1e-5, 1e-6]
CS_LABELS        = [f"{t:.0e}" for t in CS_TOLERANCES]

In [ ]:
# ── Convergence Speed Analysis ───────────────────────────────────────────────
# Ported from convergence_speed_analysis.py  (STOPPING_CONDITION_ANALYSIS.md §4)
# Sweeps tolerance ∈ {1e-3, 1e-4, 1e-5, 1e-6}
# Records steps_taken; produces box plots and stopping-time ratio charts.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CS_N_EXPERIMENTS = 100
CS_UNCERTAINTY   = 0.001
CS_N_RUNS        = 100
CS_MAX_STEPS     = 100_000
CS_DEFAULT_TOL   = 1e-3
CS_TOLERANCES    = [1e-3, 1e-4, 1e-5, 1e-6]
CS_LABELS        = [f"{t:.0e}" for t in CS_TOLERANCES]

## PUD Network

In [ ]:
print("=== Convergence Speed Analysis: PUD Network ===")
run_convergence_speed(G_pud_indexed, "PUD", output_prefix="pud")

## Tobacco Network

In [ ]:
print("=== Convergence Speed Analysis: Tobacco Network ===")
run_convergence_speed(G_tobacco_indexed, "Tobacco", output_prefix="tobacco")

# Study 2: Parameter Search

In [ ]:
# ── Parameter Search ─────────────────────────────────────────────────────────
# Ported from parameter_search.py  (STOPPING_CONDITION_ANALYSIS.md §6–7)
# Grid: tolerance × uncertainty × n_experiments
# Records steps_taken and truth_share; produces heatmaps and line plots.

PS_TOLERANCES    = [1e-3, 1e-4, 1e-5, 1e-6]
PS_UNCERTAINTIES = [0.0001, 0.001, 0.005, 0.01]
PS_N_EXPERIMENTS = [100, 200, 500]
PS_N_RUNS        = 100
PS_MAX_STEPS     = 100_000

In [ ]:
# ── Parameter Search ─────────────────────────────────────────────────────────
# Ported from parameter_search.py  (STOPPING_CONDITION_ANALYSIS.md §6–7)
# Grid: tolerance × uncertainty × n_experiments
# Records steps_taken and truth_share; produces heatmaps and line plots.

PS_TOLERANCES    = [1e-3, 1e-4, 1e-5, 1e-6]
PS_UNCERTAINTIES = [0.0001, 0.001, 0.005, 0.01]
PS_N_EXPERIMENTS = [100, 200, 500]
PS_N_RUNS        = 100
PS_MAX_STEPS     = 100_000

## PUD Network

In [ ]:
print("=== Parameter Search: PUD Network ===")
run_parameter_search(G_pud_indexed, "PUD", output_prefix="pud")

## Tobacco Network

In [ ]:
print("=== Parameter Search: Tobacco Network ===")
run_parameter_search(G_tobacco_indexed, "Tobacco", output_prefix="tobacco")

## Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))

✅ Disconnected from runtime at: 2025-10-02 11:18:40 EDT


<IPython.core.display.Javascript object>